In [1]:
import sys
from pathlib import Path
# Walk up from the notebook's location until we find the directory
# containing the `eccopy` package folder, then add it to sys.path.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "eccopy").is_dir():
        sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError("Could not find the eccopy package folder above the current directory")
print("Using eccopy from:", candidate)


Using eccopy from: /home/icornejo/EccoPy/eccopy_pkg


# EccoPy-2D-V Workflow Template

Minimal template: load a real ECCO-V (MATLAB) case, clean it, run EccoPy,
load MATLAB's real output. Comparison / metrics / plotting are left to you.

**Case folder** should contain `ECCO_IN_*.npz` (DBZ, R_km, Z_km, optionally
TEMP) and `ECCO_OUT*.npz` (ECHOTYPE, CONVECTIVITY) — plus the original
`.sh`/`.m` driver scripts, useful for confirming exact parameter values.

Three things worth knowing before you start (see README "Validation status"
for full details):
1. Real MATLAB/LROSE data commonly uses a literal `-9999.0` fill value,
   not NaN — clean it yourself (Step 2) or EccoPy will treat it as real
   (very low) reflectivity.
2. `height`/`topo` are accepted by EccoPy in **km**, not metres.
3. Sub-classification (codes 14/16/18/25/32/34/36/38) now requires
   `height`, `melt`, **and** `temp` all three together — they are no
   longer alternatives. If any one is missing, EccoPy returns
   basic-only codes (1/2/3) rather than guessing at a reduced
   sub-classification.

In [2]:
import numpy as np
import glob
import warnings

from eccopy import eccopy2d_v
from eccopy.params import WindowSpec, TextureParams, ClassificationParams


## Step 1 — Locate the case files

In [3]:
CASE_DIR = "/home/icornejo/EccoPy/Test_Cases/ECCO_V_CASE_3_SPOL"   # <-- change to your case folder

in_files = glob.glob(f"{CASE_DIR}/**/*IN*.npz", recursive=True)
out_files = sorted(glob.glob(f"{CASE_DIR}/**/*OUT*.npz", recursive=True))

print("Input file(s):", in_files)
print("Output file(s):", out_files)


Input file(s): ['/home/icornejo/EccoPy/Test_Cases/ECCO_V_CASE_3_SPOL/ECCO_IN_20220526_084500.npz']
Output file(s): ['/home/icornejo/EccoPy/Test_Cases/ECCO_V_CASE_3_SPOL/ECCO_OUT_20220526_084500.npz']


In [4]:
INPUT_FILE = in_files[0]
OUTPUT_FILE = out_files[0]   # <-- SPOL has a single output file, no TEST1/TEST6-style variants

data_in = np.load(INPUT_FILE)
data_out = np.load(OUTPUT_FILE)

print("Input fields:", data_in.files)
print("Output fields:", data_out.files)


Input fields: ['DBZ', 'LAT', 'LON', 'R_km', 'Z_km', 'TEMP']
Output fields: ['CONVECTIVITY', 'ECHOTYPE']


## Step 2 — Load and clean the data

Clean literal `-9999.0` fill values to NaN before calling EccoPy.

If you want sub-classification, you need `height`, `melt`, and `temp`
all three — see the note below on constructing `melt` if your case
doesn't provide one directly.

In [5]:
dbz = data_in['DBZ'].astype(float)
dbz[dbz == -9999.0] = np.nan   # <-- clean fill values; repeat for other arrays if needed

r_km = data_in['R_km'].astype(float)
z_km = data_in['Z_km'].astype(float)

# SPOL case: temperature, height, melt, and topo are all available/derivable --
# needed together for sub-classification (see Step 4 note).
temp_c = data_in['TEMP'].astype(float).squeeze()
temp_c[temp_c == -9999.0] = np.nan

height_km = np.broadcast_to(z_km[:, None], dbz.shape).copy()

# melt built from temp using the same convention this case's .m script uses
# (MELTING_LAYER = 20 where TEMP<=0, 10 where TEMP>0):
melt_field = np.full(temp_c.shape, np.nan)
melt_field[temp_c <= 0] = 20.0
melt_field[temp_c > 0] = 10.0

# earth-curvature beam-height correction, matching this case's .m script
# (pseudoRad = earthRadius*4/3; elAng=0 for this RHI geometry). Kept in km
# throughout -- no need to convert to metres and back like the .m script does.
pseudo_rad_km = 6371.0 * 4 / 3
topo_km = -pseudo_rad_km + np.sqrt(r_km**2 + pseudo_rad_km**2)
topo_km = np.broadcast_to(topo_km[None, :], dbz.shape).copy()


## Step 3 — Set parameters

Check the case's `.sh` script for `RADIUS` (bare pixel radius, NOT km)
and `DBZLIM` (→ `texture_limit_high`), and the `.m` script for any other
non-default values (enlargeMixed, enlargeConv, surfAltLim, etc.).

**`ClassificationParams.enlarge_conv` defaults to 3** — every case this
package has actually been validated against (SEA, SPOL) used `enlargeConv=5`
in its `.m` script, overriding that default. The default itself has not
been checked against real MATLAB output. Set `enlarge_conv` explicitly
from your case's script rather than relying on the package default.

`surf_alt_lim` lives on `ClassificationParams`, not a separate
`VerticalParams` object — `VerticalParams` is scoped to the 3-D path
only and isn't used by EccoPy-2D-V at all.

In [6]:
RADIUS = 19          # <-- from the case's run_ecco_v_Test_*.sh (SPOL: RADIUS=19)
DBZLIM = 29.0         # <-- SPOL: DBZLIM=29
ENLARGE_MIXED = 5
ENLARGE_CONV = 5     # <-- package default is 3; override to match your case's .m script
SURF_ALT_LIM = 200.0  # <-- metres; check the case's .m script

window = WindowSpec(RADIUS)
texture_params = TextureParams(texture_limit_high=DBZLIM)
class_params = ClassificationParams(
    enlarge_mixed=ENLARGE_MIXED, enlarge_conv=ENLARGE_CONV, surf_alt_lim=SURF_ALT_LIM,
)


## Step 4 — Run EccoPy

Basic classification shown below. If your case has real height/melt/temp,
uncomment and adapt the extra arguments — **all three are required
together** for sub-classification, see README re: the earth-curvature
`topo` correction several ECCO-V cases use, and the note that
`eccopy2d_v.run()` expects `height`/`topo` in **km**.

In [7]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    result = eccopy2d_v.run(
        dbz, coords_z=z_km, coords_x=r_km,
        window=window,
        texture_params=texture_params,
        class_params=class_params,
        height=height_km, melt=melt_field, temp=temp_c, topo=topo_km,
    )

our_echo = result.echo_type
our_conv = result.convectivity
our_texture = result.texture

print("echo_type shape:", our_echo.shape)
print("codes present:", sorted(set(our_echo[~np.isnan(our_echo)].astype(int).tolist())))


FileNotFoundError: No exported MATLAB disk mask for radius 5. Run export_disk_strels.m with RADII including 5 and bundle the resulting .mat file at disk_strels/disk_strel_r5.mat.

## Step 5 — MATLAB's real output

Loaded and ready for comparison.

In [ ]:
lrose_echo = data_out['ECHOTYPE']
lrose_conv = data_out['CONVECTIVITY'] if 'CONVECTIVITY' in data_out.files else None


## Your analysis

`our_echo`, `our_conv`, `our_texture` vs. `lrose_echo`, `lrose_conv` —
ready to compare, score, or plot however's useful.

Note: MATLAB's ECHOTYPE may use sub-classified codes
(14/16/18/25/30/32/34/36/38) even if you ran EccoPy in basic mode
(1/2/3) — you may want to map one down to the other before comparing,
e.g.:

```python
lrose_basic = np.zeros_like(lrose_echo)
lrose_basic[np.isin(lrose_echo, [14,16,18])] = 1
lrose_basic[lrose_echo == 25] = 2
lrose_basic[np.isin(lrose_echo, [30,32,34,36,38])] = 3
```

Code 30 is a near-aircraft override that's rarely triggered in
ground-based radar cases — safe to fold into the convective bucket
alongside 32/34/36/38 unless your case specifically involves airborne
radar.